In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import Polygon
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
# Путь к папке, где лежат твои части
parts_dir = Path('parsed_parts')

# Собираем список всех .parquet файлов
parquet_files = sorted(parts_dir.glob('*.parquet'))

# Читаем все файлы и объединяем
gdf = gpd.GeoDataFrame(
    pd.concat([gpd.read_parquet(f) for f in parquet_files], ignore_index=True),
    crs="EPSG:3857" 
)

gdf = gdf.fillna(0).to_crs(32636)
gdf.head()

,cadastralDistrictsCode,category,descr,externalKey,geom_data_id,label,options,subcategory,system_info,geometry
0,10,36368,10:12:0022203:143,10:12:0022203:143,326314297,10:12:0022203:143,"{'area': None, 'cad_num': '10:12:0022203:143',...",8,"{'inserted': '2024-10-29T00:06:15.01311', 'ins...","POLYGON ((328935.791 6829912.681, 328928.968 6..."
1,10,36368,10:12:0022201:543,10:12:0022201:543,326681245,10:12:0022201:543,"{'area': None, 'cad_num': '10:12:0022201:543',...",4,"{'inserted': '2024-10-29T01:29:40.817735', 'in...","POLYGON ((333746.675 6829493.833, 333539.204 6..."
2,10,36368,10:12:0022201:542,10:12:0022201:542,326710799,10:12:0022201:542,"{'area': None, 'cad_num': '10:12:0022201:542',...",4,"{'inserted': '2024-10-29T01:36:43.350007', 'in...","POLYGON ((329430.44 6830720, 329425.743 683073..."
3,10,36368,10:12:0022203:413,10:12:0022203:413,326658378,10:12:0022203:413,"{'area': None, 'cad_num': '10:12:0022203:413',...",4,"{'inserted': '2024-10-29T01:24:30.334365', 'in...","POLYGON ((329524.216 6832193.716, 329525.608 6..."
4,10,36368,10:12:0022203:447,10:12:0022203:447,326659948,10:12:0022203:447,"{'area': None, 'cad_num': '10:12:0022203:447',...",4,"{'inserted': '2024-10-29T01:24:48.65429', 'ins...","POLYGON ((329334.725 6832275.77, 329262.132 68..."


In [3]:
import ast
import pandas as pd


def to_dict(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    elif isinstance(x, dict):
        return x
    else:
        raise ValueError(f"Unexpected type in options: {type(x)}")

gdf['options'] = gdf['options'].apply(to_dict)

# 2) «Разворачиваем» словари в DataFrame
opts = pd.json_normalize(gdf['options'])

# 3) Склеиваем и удаляем исходный столбец
gdf = pd.concat([gdf.drop(columns='options'), opts], axis=1)
gdf.head()


,cadastralDistrictsCode,category,descr,externalKey,geom_data_id,label,subcategory,system_info,geometry,area,...,land_record_subtype,land_record_type,ownership_type,permitted_use_established_by_document,previously_posted,quarter_cad_number,readable_address,right_type,specified_area,status
0,10,36368,10:12:0022203:143,10:12:0022203:143,326314297,10:12:0022203:143,8,"{'inserted': '2024-10-29T00:06:15.01311', 'ins...","POLYGON ((328935.791 6829912.681, 328928.968 6...",NaN,...,Землепользование,Земельный участок,None,крестьянское фермерское хозяйство,None,10:12:0022203,"Республика Карелия, Лахденпохский район.Земель...",None,54704.0,Ранее учтенный
1,10,36368,10:12:0022201:543,10:12:0022201:543,326681245,10:12:0022201:543,4,"{'inserted': '2024-10-29T01:29:40.817735', 'in...","POLYGON ((333746.675 6829493.833, 333539.204 6...",NaN,...,Землепользование,Земельный участок,None,для ведения лесного хозяйства,None,10:12:0022201,"Республика Карелия, Лахденпохский район, Лахд...",None,16668560.0,Учтенный
2,10,36368,10:12:0022201:542,10:12:0022201:542,326710799,10:12:0022201:542,4,"{'inserted': '2024-10-29T01:36:43.350007', 'in...","POLYGON ((329430.44 6830720, 329425.743 683073...",NaN,...,Землепользование,Земельный участок,None,для ведения лесного хозяйства,None,10:12:0022201,"Республика Карелия, Лахденпохский район, Лахд...",None,22300.0,Учтенный
3,10,36368,10:12:0022203:413,10:12:0022203:413,326658378,10:12:0022203:413,4,"{'inserted': '2024-10-29T01:24:30.334365', 'in...","POLYGON ((329524.216 6832193.716, 329525.608 6...",NaN,...,Землепользование,Земельный участок,None,"Для эксплуатации существующих объектов ""Точка ...",None,10:12:0022203,"Республика Карелия, Лахденпохский район, кварт...",None,15633.0,Учтенный
4,10,36368,10:12:0022203:447,10:12:0022203:447,326659948,10:12:0022203:447,4,"{'inserted': '2024-10-29T01:24:48.65429', 'ins...","POLYGON ((329334.725 6832275.77, 329262.132 68...",NaN,...,Землепользование,Земельный участок,None,"Для эксплуатации существующего объекта "" Точка...",None,10:12:0022203,"Республика Карелия, Лахденпохский район, Лахд...",None,68968.0,Учтенный


In [4]:
columns_to_keep = [
    'geometry',
    'specified_area',
    'cost_index',
    'cost_value',
    'land_record_category_type',
    'permitted_use_established_by_document',
    'readable_address'
]

# Оставляем только нужные столбцы
gdf_filtered = gdf[columns_to_keep]
gdf_filtered.head()

,geometry,specified_area,cost_index,cost_value,land_record_category_type,permitted_use_established_by_document,readable_address
0,"POLYGON ((328935.791 6829912.681, 328928.968 6...",54704.0,1.81,99014.24,Земли сельскохозяйственного назначения,крестьянское фермерское хозяйство,"Республика Карелия, Лахденпохский район.Земель..."
1,"POLYGON ((333746.675 6829493.833, 333539.204 6...",16668560.0,3.21,53506077.60,Земли лесного фонда,для ведения лесного хозяйства,"Республика Карелия, Лахденпохский район, Лахд..."
2,"POLYGON ((329430.44 6830720, 329425.743 683073...",22300.0,3.21,71583.00,Земли лесного фонда,для ведения лесного хозяйства,"Республика Карелия, Лахденпохский район, Лахд..."
3,"POLYGON ((329524.216 6832193.716, 329525.608 6...",15633.0,151.45,2367657.02,Земли лесного фонда,"Для эксплуатации существующих объектов ""Точка ...","Республика Карелия, Лахденпохский район, кварт..."
4,"POLYGON ((329334.725 6832275.77, 329262.132 68...",68968.0,151.45,10445376.43,Земли лесного фонда,"Для эксплуатации существующего объекта "" Точка...","Республика Карелия, Лахденпохский район, Лахд..."


In [5]:
# Уникальные значения + сколько раз каждое встречается
print(gdf['land_record_category_type'].value_counts())


land_record_category_type
Земли населенных пунктов                                                                                                                                                                                          705467
Земли сельскохозяйственного назначения                                                                                                                                                                            544903
                                                                                                                                                                                                                   27391
Земли промышленности, энергетики, транспорта, связи, радиовещания, телевидения, информатики, земли для обеспечения космической деятельности, земли обороны, безопасности и земли иного специального назначения     22916
Земли лесного фонда                                                                                       

In [6]:
gdf_filtered = gdf[
    gdf_filtered['readable_address'].str.contains('Гатчинский район', na=False)
]

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import matplotlib.colors as mcolors
import mapclassify

# --- Подготовка данных ---
# Допустим, у тебя уже есть отфильтрованный gdf
# gdf_filtered = ...

# Цветовая палитра
colors = ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#e31a1c', '#800026']
cmap = mcolors.ListedColormap(colors)

# --- 1. Визуализация по cost_value (стоимость участка) ---
classifier_value = mapclassify.NaturalBreaks(gdf_filtered['cost_value'].dropna(), k=7)
bin_edges_value = classifier_value.bins
bin_labels_value = [f"{int(bin_edges_value[i-1]) if i > 0 else 0}–{int(b)}" for i, b in enumerate(bin_edges_value)]

gdf_filtered['Категория_стоимости_участка'] = pd.cut(
    gdf_filtered['cost_value'],
    bins=[0] + list(bin_edges_value),
    labels=bin_labels_value,
    include_lowest=True
)

# --- 2. Визуализация по cost_index (стоимость за м²) ---
classifier_index = mapclassify.NaturalBreaks(gdf_filtered['cost_index'].dropna(), k=7)
bin_edges_index = classifier_index.bins
bin_labels_index = [f"{round(bin_edges_index[i-1], 2) if i > 0 else 0}–{round(b, 2)}" for i, b in enumerate(bin_edges_index)]

gdf_filtered['Категория_стоимости_за_м2'] = pd.cut(
    gdf_filtered['cost_index'],
    bins=[0] + list(bin_edges_index),
    labels=bin_labels_index,
    include_lowest=True
)

# --- Построение карт ---
fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Карта 1 — Стоимость участка
gdf_filtered.plot(
    column='Категория_стоимости_участка',
    cmap=cmap,
    linewidth=0.2,
    edgecolor='black',
    legend=True,
    legend_kwds={'title': "Стоимость участка"},
    ax=axes[0]
)
axes[0].set_title('Стоимость участка', fontsize=24, pad=20)
axes[0].axis('off')

# Карта 2 — Стоимость за квадратный метр
gdf_filtered.plot(
    column='Категория_стоимости_за_м2',
    cmap=cmap,
    linewidth=0.2,
    edgecolor='black',
    legend=True,
    legend_kwds={'title': "Стоимость за м²"},
    ax=axes[1]
)
axes[1].set_title('Стоимость за м²', fontsize=24, pad=20)
axes[1].axis('off')

plt.tight_layout()
plt.show()
